In [23]:
import psycopg
from psycopg.types.json import Jsonb
from psycopg.rows import dict_row

In [24]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="initdb", 
    user="admin",
    password="qwer123456",
)


print("连接成功:", conn.info.server_version) 

连接成功: 170010


In [25]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id      SERIAL PRIMARY KEY, 
            name    VARCHAR(50) NOT NULL,
            profile JSONB     
        )
    """)
conn.commit()
print("建表完成")

建表完成


In [26]:
with conn.cursor() as cur:
    cur.execute(
        "INSERT INTO students (name, profile) VALUES (%s, %s)",
        ("Alice", Jsonb({"age": 20, "score": 88.5})),
    )

conn.commit() 
print("插入数据完成")

插入数据完成


In [27]:
rows = [
    ("Carol", Jsonb({"age": 19, "score": 76.0})),
    ("Dave", Jsonb({"age": 23, "score": 82.5})),
    ("Eve", Jsonb({"age": 21, "score": 95.0})),
]
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO students (name, profile) VALUES (%s, %s)",
        rows,
    )
conn.commit()
print(f"批量插入{len(rows)}条")

批量插入3条


In [28]:
with conn.cursor() as cur:
    cur.execute("SELECT id, name, profile FROM students ORDER BY id")
    rows = cur.fetchall()

    columns = [d.name for d in cur.description]
    print("列名:", columns)
    for row in rows:
        print(row)  
        print(type(row))  # tuple
        print(type(row[2]))  # dict(JSONB)
        print()

列名: ['id', 'name', 'profile']
(1, 'Alice', {'age': 20, 'score': 90.0})
<class 'tuple'>
<class 'dict'>

(2, 'Carol', {'age': 19, 'score': 76.0})
<class 'tuple'>
<class 'dict'>

(3, 'Dave', {'age': 23, 'score': 82.5})
<class 'tuple'>
<class 'dict'>

(5, 'Alice', {'age': 20, 'score': 88.5})
<class 'tuple'>
<class 'dict'>

(6, 'Carol', {'age': 19, 'score': 76.0})
<class 'tuple'>
<class 'dict'>

(7, 'Dave', {'age': 23, 'score': 82.5})
<class 'tuple'>
<class 'dict'>

(8, 'Eve', {'age': 21, 'score': 95.0})
<class 'tuple'>
<class 'dict'>



In [29]:
with conn.cursor(row_factory=dict_row) as cur:
    cur.execute("SELECT id, name, profile FROM students ORDER BY id")
    for row in cur:
        print(row)

{'id': 1, 'name': 'Alice', 'profile': {'age': 20, 'score': 90.0}}
{'id': 2, 'name': 'Carol', 'profile': {'age': 19, 'score': 76.0}}
{'id': 3, 'name': 'Dave', 'profile': {'age': 23, 'score': 82.5}}
{'id': 5, 'name': 'Alice', 'profile': {'age': 20, 'score': 88.5}}
{'id': 6, 'name': 'Carol', 'profile': {'age': 19, 'score': 76.0}}
{'id': 7, 'name': 'Dave', 'profile': {'age': 23, 'score': 82.5}}
{'id': 8, 'name': 'Eve', 'profile': {'age': 21, 'score': 95.0}}


In [30]:
with conn.cursor() as cur:
    cur.execute(
        "UPDATE students SET profile = jsonb_set(profile, '{score}', %s) WHERE name = %s",
        (Jsonb(90.0), "Alice"),
    )
    print("更新行数:", cur.rowcount)
    
    cur.execute("DELETE FROM students WHERE name = %s", ("Eve",))
    print("删除行数:", cur.rowcount)

conn.commit()

更新行数: 2
删除行数: 1


In [31]:
try:
    with conn.cursor() as cur:
        # 第一条语句正常执行
        cur.execute(
            "UPDATE students SET profile = jsonb_set(profile, '{score}', %s) WHERE name = %s",
            (Jsonb(100.0), "Alice"),
        )
        print("更新行数:", cur.rowcount)

        # 第二条语句故意出错(列age不存在),触发异常
        cur.execute("UPDATE students SET age = %s WHERE name = %s", (18, "Carol"))

    conn.commit()
except Exception as e:
    # Roll back to the start of any pending transaction.
    conn.rollback()  
    print("发生异常,已回滚:", e)


更新行数: 2
发生异常,已回滚: column "age" of relation "students" does not exist
LINE 1: UPDATE students SET age = $1 WHERE name = $2
                            ^
